In [1]:
import torch

from rlaopt.linalg import IdentityConfig, LinSys, NystromConfig
from rlaopt.solvers import PCG, PCGConfig

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
n = 10000
eigvals = torch.arange(1, n + 1) ** -2.0
reg = 1e-3

U = torch.randn(n, n)
U = torch.linalg.qr(U).Q

A = U @ torch.diag(eigvals) @ U.T
B = torch.randn(n, 10)

In [4]:
lin_sys = LinSys(A, B, reg)
lin_sys.cuda()

LinSys()

In [5]:
preconditioner_config_identity = IdentityConfig()
preconditioner_config_nystrom = NystromConfig(rank=100, base_damping=reg)

In [6]:
solver_config = PCGConfig(
    max_iters=500,
    tol=1e-8,
    preconditioner_config=preconditioner_config_nystrom,
)

In [7]:
solver = PCG(solver_config)
state = solver.init_state(lin_sys)

In [8]:
max_iters = 100

for i in range(max_iters):
    state = solver.step(lin_sys, state)
    print(f"Iteration {i + 1}, residual norm: {state.res_norm}")

Iteration 1, residual norm: tensor([1.9356, 1.9072, 2.1289, 1.7084, 1.9007, 1.6910, 1.4831, 1.7878, 1.6143,
        1.7497], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 2, residual norm: tensor([0.1243, 0.1696, 0.1986, 0.1412, 0.1289, 0.1386, 0.1126, 0.1245, 0.1228,
        0.1262], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 3, residual norm: tensor([0.0096, 0.0119, 0.0125, 0.0101, 0.0085, 0.0106, 0.0082, 0.0070, 0.0096,
        0.0082], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 4, residual norm: tensor([0.0007, 0.0006, 0.0008, 0.0008, 0.0005, 0.0008, 0.0004, 0.0005, 0.0006,
        0.0004], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 5, residual norm: tensor([3.4612e-05, 2.8215e-05, 3.0450e-05, 4.0628e-05, 2.6833e-05, 3.5176e-05,
        2.3543e-05, 2.4620e-05, 2.9952e-05, 1.9900e-05], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 6, residual norm: tensor([1.4217e-06, 1.1958e-06